In [212]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
print('All packages loaded')

All packages loaded


# Introduction
In this notebook we calibrate a number of models on EURIBOR daily data, with the aim to compare their performance with a simple Random Walk model. The models that we are going to use can be categorized in 2 different field of study:
1. **Mathematical Finance**: Vasicek, CIR
2. **Machine Learning**: ARMA, Random Forest, LSTM

For each model we perform a **rolling window** approach, with a number of **30** observations for each step, and forecast: 
*1-day, 1_week, 1_month, 3_months ahead*. 

# Import EURIBOR data
First step is importing the full dataset, clean it, and keep only the time series we are working on.

In [213]:
df = pd.read_excel('EURIBOR.xlsx')
df.head()

,Date,1 week,1 month,3 month,6 month,12 month
0,2006-04-03,2.614,2.647,2.818,2.992,3.254
1,2006-04-04,2.615,2.649,2.822,3.002,3.264
2,2006-04-05,2.635,2.651,2.824,3.006,3.260
3,2006-04-06,2.653,2.654,2.831,3.007,3.260
4,2006-04-07,2.643,2.641,2.764,2.915,3.159


Let's keep the 1 week maturity rates only, which are the best proxy for **short term rates**, and keep the Date as a singular Series. This is important as the Mathematical Finance models we use, have been theoretically built to model short term rates in continuous time. 

In [215]:
dates = df.Date
euribor_ir = df.iloc[:, 1] 
euribor_ir.head()

0    2.614
1    2.615
2    2.635
3    2.653
4    2.643
Name: 1 week, dtype: float64

# Random Walk 
This simple model can be mathematically expressed as follows:

$i_{t+\delta t} = i_{t} + \epsilon$

Where $\epsilon$ is a disturbance term with zero mean. 
This allows us to employ the **Random Walk** as a straightforward benchmark, where the optimal prediction for rates at time $t+1$ is merely the rates at time $t$. In essence, our time series is inherently unpredictable, as there is no additional information that can be harnessed for forecasting purposes. 

In [255]:
df = euribor_ir.copy()
df = pd.DataFrame({'series1': df})
df = df.join(euribor_ir.shift(1),  how='left')
df.columns = ['rates t', 'rates t-1']
df = df.dropna()
df.reset_index(inplace=True)
df = df.iloc[:, 1:]
print(df.head())


# split y and X
y = df[['rates t']]
X = df[['rates t-1']]

   rates t  rates t-1
0    2.615      2.614
1    2.635      2.615
2    2.653      2.635
3    2.643      2.653
4    2.630      2.643


In [256]:
# Variables of interest for the calibration
ts_length = df.shape[0]
n_obs = 30
max_time_step = 90

pred1_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred7_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred30_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred90_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact1_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact7_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact30_v = np.zeros(ts_length-(n_obs+max_time_step-1))
exact90_v = np.zeros(ts_length-(n_obs+max_time_step-1))

"""
create a for loop for the rolling window
we subtract n_obs because n is the first index of the sample, not the last one
I need to train the first 30 obs, then 2:31 etc... 
"""
for n in range(ts_length-n_obs):

    # by doing that I keep all the observations from n to (n+n_obs-1)
    X_train = X[n:n+n_obs]
    y_train = y[n:n+n_obs]
    prediction_index = y_train.index[-1]
    
    # Need to break the loop if my max prediction cannot be backtested
    if (prediction_index + max_time_step) > y.index[-1]:
        break
    else:
        # predictions and exact values at 4 time steps 
        r0 = y.iloc[prediction_index]

        pred1 = r0 
        pred7 = r0 
        pred30 = r0 
        pred90 = r0 

        exact1 = y.iloc[prediction_index + 1]
        exact7 = y.iloc[prediction_index + 7]
        exact30 = y.iloc[prediction_index + 30]
        exact90 = y.iloc[prediction_index + 90]

        # store the predictions and the exact values
        pred1_v[n] = pred1
        pred7_v[n] = pred7
        pred30_v[n] = pred30
        pred90_v[n] = pred90
        
        exact1_v[n] = exact1
        exact7_v[n] = exact7
        exact30_v[n] = exact30
        exact90_v[n] = exact90

In [257]:
# Create a dataframe with all the predictions and one with all exact values
pred_RW = {
    '1 step pred': pred1_v,
    '7 step pred': pred7_v,
    '30 step pred': pred30_v,
    '90 step pred': pred90_v
}

exact_rates = {
    '1 step exact': exact1_v,
    '7 step exact': exact7_v,
    '30 step exact': exact30_v,
    '90 step exact': exact90_v   
}

pred_RW = pd.DataFrame(pred_RW)
exact_rates = pd.DataFrame(exact_rates)
pred_RW.index = dates.iloc[1:pred_RW.shape[0]+1]
exact_rates.index = dates.iloc[1:pred_RW.shape[0]+1]

In [267]:
# Compute MSE
SE_RW = pd.DataFrame(columns=['1 step ahead', '7 steps ahead', '30 steps ahead', '90 steps ahead'])
for n in range(4):
    SE_RW.iloc[:, n] = (pred_RW.iloc[:, n] - exact_rates.iloc[:, n])**2

SE_RW.head()

C:\Users\yuria\AppData\Local\Temp\ipykernel_22088\2900373717.py:4: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  SE_RW.iloc[:, n] = (pred_RW.iloc[:, n] - exact_rates.iloc[:, n])**2


,1 step ahead,7 steps ahead,30 steps ahead,90 steps ahead
Date,,,,
2006-04-04,0.000000e+00,0.000009,0.053824,0.192721
2006-04-05,0.000000e+00,0.000036,0.049729,0.190096
2006-04-06,2.500000e-05,0.000004,0.047089,0.189225
2006-04-07,4.000000e-06,0.000049,0.041616,0.186624
2006-04-10,1.000000e-06,0.000196,0.038809,0.213444


# Vasicek Model

## Test with 30 observations
The next step is running a regression. Let's start by keeping 30 days and working with that. The objective is to make sure that the calibrated parameters make sense, in particular $\theta$ has to be coherent with the data, and that Monte Carlo predictions matches with the analytical ones.

In [109]:
# keep 30 days to test the algo
trial = df.iloc[:30, :]

# split y and X
y = trial[['rates t']]
X = trial[['rates t-1']]

# Build the model
model = LinearRegression()
model.fit(X, y)

# Get the estimated parameters and dt
intercept = model.intercept_  # Intercept (bias)
slope = model.coef_[0]        # Slope (coefficients)
dt = 1/252                    # since yield are in annual term

# Calculate the predicted values (y_hat) and the residuals
y_hat = model.predict(X)
residuals = y - y_hat

# get the parameter of interest for Vasicek
k = (1-intercept)/dt
theta = slope / (1-intercept)
sigma = np.std(residuals)/dt

# see the calibrated model
print(f'The calibrated Vasicek model is dr = {k}({theta}-r)+{sigma.values[0]}dW')

"""
################## SIMULATE THE MODEL 
# Initial interest rate
r0 = prova.iloc[-1, 0]

# Number of time steps
n_steps = 90

# Initialize arrays to store interest rate paths
interest_rates = np.zeros(n_steps+1)
interest_rates[0] = r0

# Simulate future interest rate path using Euler's method
for i in range(n_steps):
        dr = k * (theta - interest_rates[i]) * dt + sigma*np.sqrt(dt) * np.random.normal(0,1)
        interest_rates[i+1] = interest_rates[i] + dr

# Time steps
t = np.arange(n_steps+1)

print(f'my theta is {theta} and my initial rate is {r0}')
"""

The calibrated Vasicek model is dr = [58.53571009]([3.04780934]-r)+2.720867746272264dW


"\n################## SIMULATE THE MODEL \n# Initial interest rate\nr0 = prova.iloc[-1, 0]\n\n# Number of time steps\nn_steps = 90\n\n# Initialize arrays to store interest rate paths\ninterest_rates = np.zeros(n_steps+1)\ninterest_rates[0] = r0\n\n# Simulate future interest rate path using Euler's method\nfor i in range(n_steps):\n        dr = k * (theta - interest_rates[i]) * dt + sigma*np.sqrt(dt) * np.random.normal(0,1)\n        interest_rates[i+1] = interest_rates[i] + dr\n\n# Time steps\nt = np.arange(n_steps+1)\n\nprint(f'my theta is {theta} and my initial rate is {r0}')\n"

In [110]:
# Let's make a simulation with 90 step ahead, from that we can extract the mean at the time step we want
num_simulations = 10000
n_steps = 90
r0 = trial.iloc[-1, 0]

# Initialize an array to store simulated interest rate paths
interest_rate_paths = np.zeros((num_simulations, n_steps+1))
interest_rate_paths[:,0] = r0

# Perform Monte Carlo simulations
dW = np.random.normal(0, np.sqrt(dt), (num_simulations, n_steps))
for s in range(n_steps):
    dr = k * (theta - interest_rate_paths[:, s]) * dt + sigma.values[0] * dW[:, s]
    interest_rate_paths[:, s + 1] = interest_rate_paths[:, s] + dr

# let's take our mean estimates
simulated1 = np.mean(interest_rate_paths[:, 1])
simulated7 = np.mean(interest_rate_paths[:, 7])
simulated30 = np.mean(interest_rate_paths[:, 30])
simulated90 = np.mean(interest_rate_paths[:, 90])
print(f'the predictions using montecarlo are: {simulated1, simulated7, simulated30, simulated90}')

# let's compute them analitically
analytical1 = r0 * np.exp(-k*dt) + theta * (1 - np.exp(-k*dt))
analytical7 = r0 * np.exp(-k*7*dt) + theta * (1 - np.exp(-k*7*dt))
analytical30 = r0 * np.exp(-k*30*dt) + theta * (1 - np.exp(-k*30*dt))
analytical90 = r0 * np.exp(-k*90*dt) + theta * (1 - np.exp(-k*90*dt))
print(f'the analytical predictions are: {analytical1[0], analytical7[0], analytical30[0], analytical90[0]}')

# that's the proof we can use the analytical ones!!!

the predictions using montecarlo are: (2.7244818567568885, 2.982051631914818, 3.0446777889184102, 3.0511968495724386)
the analytical predictions are: (2.7110542635639066, 2.964242359850502, 3.047409586546365, 3.0478093439787446)


## Rolling Window Calibration
Let's now generalise the previous code using a rolling window approach. We also need to store the calibrated parameters and show them graphically.

In [259]:
# Variables of interest for the calibration
pred1_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred7_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred30_v = np.zeros(ts_length-(n_obs+max_time_step-1))
pred90_v = np.zeros(ts_length-(n_obs+max_time_step-1))

In [260]:
"""
create a for loop for the rolling window
we subtract n_obs because n is the first index of the sample, not the last one
I need to train the first 30 obs, then 2:31 etc... until ts_length-120:ts_length-90, 
since my longest 3 months prediction will fall on my last available data
"""

for n in range(ts_length-n_obs):
    #choose the model
    model = LinearRegression() 

    # by doing that I keep all the observations from n to (n+n_obs-1)
    X_train = X[n:n+n_obs]
    y_train = y[n:n+n_obs]
    prediction_index = y_train.index[-1]
    
    if (prediction_index + 90) > y.index[-1]:
        break
    else:
        # fit the model
        model.fit(X_train, y_train)

        # get parameters of the linear regression
        intercept = model.intercept_  
        slope = model.coef_[0]        

        # get the parameter of interest for Vasicek
        dt = 1/252                    
        k = (1-intercept)/dt
        theta = slope / (1-intercept)

        # predictions and exact values at 4 time steps 
        r0 = y.iloc[prediction_index]

        pred1 = r0 * np.exp(-k*dt) + theta * (1 - np.exp(-k*dt))
        pred7 = r0 * np.exp(-k*7*dt) + theta * (1 - np.exp(-k*7*dt))
        pred30 = r0 * np.exp(-k*30*dt) + theta * (1 - np.exp(-k*30*dt))
        pred90 = r0 * np.exp(-k*90*dt) + theta * (1 - np.exp(-k*90*dt))

        # store the predictions and the exact values
        pred1_v[n] = pred1
        pred7_v[n] = pred7
        pred30_v[n] = pred30
        pred90_v[n] = pred90

In [261]:
# Create a dataframe with all the predictions and one with all exact values
pred_vsk = {
    '1 step pred': pred1_v,
    '7 step pred': pred7_v,
    '30 step pred': pred30_v,
    '90 step pred': pred90_v
}

pred_VSK = pd.DataFrame(pred_vsk)
pred_VSK.index = dates.iloc[1:pred_RW.shape[0]+1]

In [266]:
# Compute MSE
SE_VSK = pd.DataFrame(columns=['1 step ahead', '7 steps ahead', '30 steps ahead', '90 steps ahead'])
for n in range(4):
    SE_VSK.iloc[:, n] = (pred_VSK.iloc[:, n] - exact_rates.iloc[:, n])**2

SE_VSK.head()

C:\Users\yuria\AppData\Local\Temp\ipykernel_22088\1666497103.py:4: DeprecationWarning: In a future version, `df.iloc[:, i] = newvals` will attempt to set the values inplace instead of always setting a new array. To retain the old behavior, use either `df[df.columns[i]] = newvals` or, if columns are non-unique, `df.isetitem(i, newvals)`
  SE_VSK.iloc[:, n] = (pred_VSK.iloc[:, n] - exact_rates.iloc[:, n])**2


,1 step ahead,7 steps ahead,30 steps ahead,90 steps ahead
Date,,,,
2006-04-04,0.007754,0.114408,0.037021,0.000201
2006-04-05,0.013663,0.212288,0.138559,0.025629
2006-04-06,0.002883,0.027707,0.163325,0.387080
2006-04-07,0.004992,0.071916,0.013502,0.012448
2006-04-10,0.000886,0.014163,0.005175,0.113494


# CIR

# ARMA

# Random Forest

# LSTM